# 🔄 Design Pattern: Multi-Agent Workflow Parallelization via Task Queue

## Pattern Overview

This notebook illustrates the **Task Queue Pattern** for parallelizing multi-agent workflows. This is a skeleton implementation demonstrating the pattern structure.

## Pattern Intent

Enable multiple worker agents to process tasks in parallel by decoupling task production from task consumption through a shared queue.

## Pattern Structure

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    TASK QUEUE PARALLELIZATION PATTERN                   │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   ┌─────────────────┐                    ┌─────────────────────┐       │
│   │    PRODUCER     │      push()        │     TASK QUEUE      │       │
│   │  (Orchestrator) │ ─────────────────► │  ┌───┬───┬───┬───┐  │       │
│   │                 │                    │  │T1 │T2 │T3 │...│  │       │
│   │  • Decompose    │                    │  └───┴───┴───┴───┘  │       │
│   │  • Create tasks │                    │                     │       │
│   │  • Push to queue│                    │  Thread-safe FIFO   │       │
│   └─────────────────┘                    └──────────┬──────────┘       │
│                                                     │                   │
│                                          pop()     │                   │
│                         ┌───────────────────────────┼────────────┐      │
│                         │                           │            │      │
│                         ▼                           ▼            ▼      │
│                 ┌─────────────┐          ┌─────────────┐  ┌───────────┐│
│                 │  CONSUMER   │          │  CONSUMER   │  │ CONSUMER  ││
│                 │  (Worker 1) │          │  (Worker 2) │  │ (Worker N)││
│                 │             │          │             │  │           ││
│                 │  • Pop task │          │  • Pop task │  │ • Pop task││
│                 │  • Execute  │          │  • Execute  │  │ • Execute ││
│                 │  • Report   │          │  • Report   │  │ • Report  ││
│                 └──────┬──────┘          └──────┬──────┘  └─────┬─────┘│
│                        │                        │                │      │
│                        └────────────────────────┼────────────────┘      │
│                                                 ▼                       │
│                                      ┌─────────────────┐                │
│                                      │   AGGREGATOR    │                │
│                                      │  • Collect      │                │
│                                      │  • Synthesize   │                │
│                                      └─────────────────┘                │
└─────────────────────────────────────────────────────────────────────────┘
```

## Key Participants

| Participant | Role |
|-------------|------|
| **Task** | Unit of work to be processed |
| **TaskQueue** | Thread-safe buffer between producer and consumers |
| **Producer (Orchestrator)** | Creates tasks and pushes them to the queue |
| **Consumer (Worker)** | Pops tasks from queue and executes them |
| **Aggregator** | Collects and synthesizes results |

## When to Use

- Tasks can be processed independently
- Work needs to be distributed across multiple workers
- Want decoupling between task creation and execution
- Need load balancing across workers

---

## Part 1: Core Data Structures

### 1.1 Task Status Enumeration

The task lifecycle states.

In [ ]:
from enum import Enum
from typing import Any, Optional, List, Dict
from dataclasses import dataclass, field
from queue import Queue
from threading import Lock
from abc import ABC, abstractmethod


class TaskStatus(Enum):
    """
    Enumeration of possible task states in the lifecycle.
    
    State Transitions:
        PENDING → IN_PROGRESS → COMPLETED
                             → FAILED
    """
    PENDING = "pending"          # Task created, waiting in queue
    IN_PROGRESS = "in_progress"  # Task being processed by a worker
    COMPLETED = "completed"      # Task finished successfully
    FAILED = "failed"            # Task encountered an error

### 1.2 Task Data Class

Represents a unit of work to be processed.

In [ ]:
@dataclass
class Task:
    """
    Represents a single unit of work in the task queue pattern.
    
    This is the data object that flows through the system:
    Producer creates → Queue holds → Consumer processes
    
    Attributes:
        task_id: Unique identifier for tracking
        payload: The actual work data (query, parameters, etc.)
        status: Current state in the lifecycle
        result: Output after processing (None until completed)
        error: Error message if failed (None if successful)
        metadata: Optional additional tracking information
    
    Usage:
        task = Task(task_id="001", payload={"query": "search term"})
        queue.push(task)
    """
    task_id: str
    payload: Any                                    # The work to be done
    status: TaskStatus = TaskStatus.PENDING
    result: Optional[Any] = None                    # Filled by worker
    error: Optional[str] = None                     # Filled on failure
    metadata: Dict[str, Any] = field(default_factory=dict)
    
    def mark_in_progress(self, worker_id: str) -> None:
        """Mark task as being processed by a worker."""
        self.status = TaskStatus.IN_PROGRESS
        self.metadata["worker_id"] = worker_id
    
    def mark_completed(self, result: Any) -> None:
        """Mark task as successfully completed with result."""
        self.status = TaskStatus.COMPLETED
        self.result = result
    
    def mark_failed(self, error: str) -> None:
        """Mark task as failed with error message."""
        self.status = TaskStatus.FAILED
        self.error = error

---

## Part 2: The Task Queue

The central component that enables the producer-consumer pattern.

In [ ]:
class TaskQueue:
    """
    Thread-safe task queue implementing the producer-consumer buffer.
    
    This is the central coordination point of the pattern:
    - Producers (orchestrators) push tasks
    - Consumers (workers) pop tasks
    - Results are collected for aggregation
    
    Key Properties:
    - Thread-safe: Multiple workers can pop concurrently
    - FIFO ordering: First task pushed is first task popped
    - Non-blocking pop: Returns None if queue is empty
    
    Pattern Role:
    ┌──────────┐     push()      ┌───────────┐     pop()      ┌──────────┐
    │ Producer │ ──────────────► │ TaskQueue │ ──────────────► │ Consumer │
    └──────────┘                 └───────────┘                 └──────────┘
    """
    
    def __init__(self):
        """Initialize the queue with thread-safety primitives."""
        self._queue: Queue = Queue()           # Thread-safe FIFO queue
        self._lock: Lock = Lock()              # Protects shared state
        self._tasks: Dict[str, Task] = {}      # Track all tasks by ID
        self._results: List[Dict] = []         # Completed results
    
    # ─────────────────────────────────────────────────────────────────────
    # PRODUCER METHODS - Called by Orchestrator
    # ─────────────────────────────────────────────────────────────────────
    
    def push(self, task: Task) -> None:
        """
        Push a task to the queue (Producer operation).
        
        Args:
            task: The task to add to the queue
            
        Thread Safety:
            Uses lock to protect _tasks dictionary update
        """
        with self._lock:
            self._tasks[task.task_id] = task
        self._queue.put(task)
        # TODO: Add logging or callback for monitoring
    
    # ─────────────────────────────────────────────────────────────────────
    # CONSUMER METHODS - Called by Workers
    # ─────────────────────────────────────────────────────────────────────
    
    def pop(self) -> Optional[Task]:
        """
        Pop a task from the queue (Consumer operation).
        
        Returns:
            Task if available, None if queue is empty
            
        Note:
            Non-blocking - returns immediately if no tasks
        """
        try:
            return self._queue.get_nowait()
        except Exception:
            return None
    
    def report_result(self, task_id: str, worker_id: str, result: Any) -> None:
        """
        Report successful task completion.
        
        Args:
            task_id: ID of the completed task
            worker_id: ID of the worker that completed it
            result: The task output
        """
        with self._lock:
            if task_id in self._tasks:
                self._tasks[task_id].mark_completed(result)
            self._results.append({
                "task_id": task_id,
                "worker_id": worker_id,
                "result": result,
                "status": "completed"
            })
    
    def report_failure(self, task_id: str, worker_id: str, error: str) -> None:
        """
        Report task failure.
        
        Args:
            task_id: ID of the failed task
            worker_id: ID of the worker that encountered the error
            error: Error message describing the failure
        """
        with self._lock:
            if task_id in self._tasks:
                self._tasks[task_id].mark_failed(error)
            self._results.append({
                "task_id": task_id,
                "worker_id": worker_id,
                "error": error,
                "status": "failed"
            })
    
    # ─────────────────────────────────────────────────────────────────────
    # QUERY METHODS - Called by Workflow Controller
    # ─────────────────────────────────────────────────────────────────────
    
    def is_empty(self) -> bool:
        """Check if there are no more tasks to process."""
        return self._queue.empty()
    
    def size(self) -> int:
        """Get number of tasks waiting in queue."""
        return self._queue.qsize()
    
    def get_all_results(self) -> List[Dict]:
        """Get all completed results for aggregation."""
        with self._lock:
            return self._results.copy()
    
    def get_stats(self) -> Dict[str, int]:
        """Get queue statistics for monitoring."""
        with self._lock:
            return {
                "total": len(self._tasks),
                "pending": self._queue.qsize(),
                "completed": sum(1 for t in self._tasks.values() 
                               if t.status == TaskStatus.COMPLETED),
                "failed": sum(1 for t in self._tasks.values() 
                            if t.status == TaskStatus.FAILED),
            }

---

## Part 3: Producer (Orchestrator)

The component that creates tasks and pushes them to the queue.

In [ ]:
class Producer(ABC):
    """
    Abstract base class for task producers (Orchestrators).
    
    The Producer is responsible for:
    1. Analyzing input (e.g., user query)
    2. Decomposing work into independent tasks
    3. Pushing tasks to the shared queue
    
    Pattern Role:
        Producer creates tasks and feeds the queue.
        Workers don't need to know how tasks are created.
    
    Subclass this to implement domain-specific task creation logic.
    """
    
    def __init__(self, queue: TaskQueue):
        """
        Initialize producer with reference to shared queue.
        
        Args:
            queue: The TaskQueue to push tasks to
        """
        self.queue = queue
        self._task_counter = 0
    
    @abstractmethod
    def decompose(self, input_data: Any) -> List[Dict]:
        """
        Decompose input into task payloads.
        
        Args:
            input_data: The input to break down (e.g., user query)
            
        Returns:
            List of task payload dictionaries
            
        IMPLEMENT THIS: Define how to split work into tasks.
        """
        pass
    
    def produce(self, input_data: Any) -> int:
        """
        Main producer method: decompose input and push tasks.
        
        Args:
            input_data: The input to process
            
        Returns:
            Number of tasks created
        """
        payloads = self.decompose(input_data)
        
        for payload in payloads:
            self._task_counter += 1
            task = Task(
                task_id=f"task_{self._task_counter:04d}",
                payload=payload
            )
            self.queue.push(task)
        
        return len(payloads)


# ─────────────────────────────────────────────────────────────────────────────
# EXAMPLE: Concrete Producer Implementation
# ─────────────────────────────────────────────────────────────────────────────

class QueryDecomposer(Producer):
    """
    Example producer that decomposes a query into search tasks.
    
    This is a skeleton - in real implementation, you might use
    an LLM to intelligently decompose the query.
    """
    
    def decompose(self, input_data: Any) -> List[Dict]:
        """
        Decompose a user query into search sub-tasks.
        
        Args:
            input_data: User query string or dict with 'query' key
            
        Returns:
            List of search task payloads
        """
        query = input_data if isinstance(input_data, str) else input_data.get("query", "")
        
        # SKELETON: Replace with actual decomposition logic
        # In real implementation, use LLM to create sub-queries
        sub_queries = [
            f"{query} - overview",
            f"{query} - recent developments",
            f"{query} - best practices",
        ]
        
        return [{"search_query": sq} for sq in sub_queries]

## Part 4: Consumer (Worker)

The component that pops tasks from the queue and executes them.

In [ ]:
class Consumer(ABC):
    """
    Abstract base class for task consumers (Workers).
    
    The Consumer is responsible for:
    1. Popping tasks from the queue
    2. Executing the task work
    3. Reporting results back to the queue
    
    Pattern Role:
        Consumer processes tasks without knowing how they were created.
        Multiple consumers can work in parallel on the same queue.
    
    Subclass this to implement domain-specific task execution logic.
    """
    
    def __init__(self, worker_id: str, queue: TaskQueue):
        """
        Initialize consumer with ID and reference to shared queue.
        
        Args:
            worker_id: Unique identifier for this worker
            queue: The TaskQueue to pop tasks from
        """
        self.worker_id = worker_id
        self.queue = queue
    
    @abstractmethod
    def execute(self, payload: Any) -> Any:
        """
        Execute the task work.
        
        Args:
            payload: The task payload containing work data
            
        Returns:
            The result of the task execution
            
        Raises:
            Exception: If task execution fails
            
        IMPLEMENT THIS: Define how to process a task.
        """
        pass
    
    def consume_one(self) -> Optional[Dict]:
        """
        Consume and process one task from the queue.
        
        Returns:
            Result dict if task was processed, None if queue was empty
        """
        task = self.queue.pop()
        
        if task is None:
            return None  # Queue is empty
        
        task.mark_in_progress(self.worker_id)
        
        try:
            result = self.execute(task.payload)
            self.queue.report_result(task.task_id, self.worker_id, result)
            return {"task_id": task.task_id, "status": "completed", "result": result}
            
        except Exception as e:
            self.queue.report_failure(task.task_id, self.worker_id, str(e))
            return {"task_id": task.task_id, "status": "failed", "error": str(e)}
    
    def consume_all(self) -> List[Dict]:
        """
        Consume all available tasks from the queue.
        
        Returns:
            List of result dicts for all processed tasks
        """
        results = []
        while not self.queue.is_empty():
            result = self.consume_one()
            if result:
                results.append(result)
        return results


# ─────────────────────────────────────────────────────────────────────────────
# EXAMPLE: Concrete Consumer Implementation
# ─────────────────────────────────────────────────────────────────────────────

class SearchWorker(Consumer):
    """
    Example consumer that executes search tasks.
    
    This is a skeleton - in real implementation, you would
    call an actual search API (e.g., Tavily, Google, etc.).
    """
    
    def execute(self, payload: Any) -> Any:
        """
        Execute a search task.
        
        Args:
            payload: Dict with 'search_query' key
            
        Returns:
            Search results (simulated in skeleton)
        """
        query = payload.get("search_query", "")
        
        # SKELETON: Replace with actual search implementation
        # In real implementation, call search API here
        return {
            "query": query,
            "results": [f"Result 1 for: {query}", f"Result 2 for: {query}"],
            "source": "skeleton_search"
        }

## Part 5: Aggregator

The component that collects and synthesizes results from all workers.

In [ ]:
class Aggregator(ABC):
    """
    Abstract base class for result aggregators.
    
    The Aggregator is responsible for:
    1. Collecting results from completed tasks
    2. Synthesizing results into a final output
    
    Pattern Role:
        Aggregator combines worker outputs into unified result.
        Called after all tasks are processed.
    
    Subclass this to implement domain-specific synthesis logic.
    """
    
    def __init__(self, queue: TaskQueue):
        """
        Initialize aggregator with reference to shared queue.
        
        Args:
            queue: The TaskQueue to collect results from
        """
        self.queue = queue
    
    @abstractmethod
    def synthesize(self, results: List[Dict]) -> Any:
        """
        Synthesize multiple results into a final output.
        
        Args:
            results: List of result dicts from workers
            
        Returns:
            Synthesized final output
            
        IMPLEMENT THIS: Define how to combine results.
        """
        pass
    
    def aggregate(self) -> Any:
        """
        Main aggregation method: collect results and synthesize.
        
        Returns:
            Final synthesized output
        """
        results = self.queue.get_all_results()
        successful_results = [r for r in results if r.get("status") == "completed"]
        return self.synthesize(successful_results)


# ─────────────────────────────────────────────────────────────────────────────
# EXAMPLE: Concrete Aggregator Implementation
# ─────────────────────────────────────────────────────────────────────────────

class ResultSynthesizer(Aggregator):
    """
    Example aggregator that combines search results.
    
    This is a skeleton - in real implementation, you might use
    an LLM to intelligently synthesize the results.
    """
    
    def synthesize(self, results: List[Dict]) -> Any:
        """
        Synthesize search results into a report.
        
        Args:
            results: List of search result dicts
            
        Returns:
            Combined report dict
        """
        # SKELETON: Replace with actual synthesis logic
        # In real implementation, use LLM to create summary
        all_findings = []
        for r in results:
            if "result" in r and "results" in r["result"]:
                all_findings.extend(r["result"]["results"])
        
        return {
            "summary": "Combined results from all searches",
            "total_results": len(all_findings),
            "findings": all_findings,
            "stats": self.queue.get_stats()
        }

## Part 6: Workflow Coordinator

The component that orchestrates the entire pattern execution.

In [ ]:
class WorkflowCoordinator:
    """
    Coordinates the task queue parallelization workflow.
    
    This is the "glue" that ties the pattern together:
    1. Creates the shared queue
    2. Runs the producer to create tasks
    3. Runs workers to process tasks (with configurable parallelism)
    4. Runs the aggregator to synthesize results
    
    Workflow:
        ┌──────────────┐     ┌───────────┐     ┌─────────────┐     ┌────────────┐
        │   Producer   │ ──► │   Queue   │ ──► │   Workers   │ ──► │ Aggregator │
        │  (1x run)    │     │  (shared) │     │ (N workers) │     │  (1x run)  │
        └──────────────┘     └───────────┘     └─────────────┘     └────────────┘
    """
    
    def __init__(
        self,
        producer: Producer,
        consumer_factory,  # Callable that creates Consumer instances
        aggregator: Aggregator,
        num_workers: int = 3
    ):
        """
        Initialize the workflow coordinator.
        
        Args:
            producer: The Producer instance (orchestrator)
            consumer_factory: Factory function (worker_id, queue) -> Consumer
            aggregator: The Aggregator instance
            num_workers: Number of parallel workers
        """
        self.producer = producer
        self.consumer_factory = consumer_factory
        self.aggregator = aggregator
        self.num_workers = num_workers
        self.queue = producer.queue  # Shared queue
    
    def run(self, input_data: Any) -> Any:
        """
        Execute the complete workflow.
        
        Args:
            input_data: Input to the producer (e.g., user query)
            
        Returns:
            Final aggregated result
            
        Workflow Steps:
            1. Producer decomposes input and pushes tasks
            2. Workers pop and process tasks in rounds
            3. Aggregator collects and synthesizes results
        """
        # ─────────────────────────────────────────────────────────────────────
        # PHASE 1: Production - Create and queue tasks
        # ─────────────────────────────────────────────────────────────────────
        print(f"📋 PRODUCER: Creating tasks...")
        num_tasks = self.producer.produce(input_data)
        print(f"   Created {num_tasks} tasks")
        
        # ─────────────────────────────────────────────────────────────────────
        # PHASE 2: Consumption - Workers process tasks
        # ─────────────────────────────────────────────────────────────────────
        print(f"\n👷 WORKERS: Processing tasks with {self.num_workers} workers...")
        
        # Create worker instances
        workers = [
            self.consumer_factory(f"worker_{i+1}", self.queue)
            for i in range(self.num_workers)
        ]
        
        # Process in rounds until queue is empty
        round_num = 0
        while not self.queue.is_empty():
            round_num += 1
            print(f"\n   Round {round_num}:")
            
            for worker in workers:
                if self.queue.is_empty():
                    break
                result = worker.consume_one()
                if result:
                    status = "✅" if result["status"] == "completed" else "❌"
                    print(f"      {status} {worker.worker_id}: {result['task_id']}")
        
        print(f"\n   Queue stats: {self.queue.get_stats()}")
        
        # ─────────────────────────────────────────────────────────────────────
        # PHASE 3: Aggregation - Synthesize results
        # ─────────────────────────────────────────────────────────────────────
        print(f"\n📊 AGGREGATOR: Synthesizing results...")
        final_result = self.aggregator.aggregate()
        print(f"   Done!")
        
        return final_result

---

## Part 7: Usage Example

Putting all the components together.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# USAGE EXAMPLE: Running the Task Queue Parallelization Pattern
# ═══════════════════════════════════════════════════════════════════════════════

def run_example():
    """
    Demonstrates the complete task queue parallelization pattern.
    """
    print("=" * 70)
    print("TASK QUEUE PARALLELIZATION PATTERN - DEMO")
    print("=" * 70)
    
    # Step 1: Create the shared queue
    queue = TaskQueue()
    
    # Step 2: Create pattern components
    producer = QueryDecomposer(queue)
    aggregator = ResultSynthesizer(queue)
    
    # Step 3: Define consumer factory
    def create_worker(worker_id, q):
        return SearchWorker(worker_id, q)
    
    # Step 4: Create coordinator
    coordinator = WorkflowCoordinator(
        producer=producer,
        consumer_factory=create_worker,
        aggregator=aggregator,
        num_workers=3
    )
    
    # Step 5: Run the workflow
    result = coordinator.run("artificial intelligence trends")
    
    # Step 6: Display result
    print("\n" + "=" * 70)
    print("FINAL RESULT")
    print("=" * 70)
    print(f"Summary: {result['summary']}")
    print(f"Total findings: {result['total_results']}")
    print(f"Stats: {result['stats']}")
    print("\nFindings:")
    for i, finding in enumerate(result['findings'], 1):
        print(f"  {i}. {finding}")
    
    return result


# Run the example
example_result = run_example()

---

## Part 8: Pattern Summary

### Class Diagram

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                              CLASS DIAGRAM                                  │
└─────────────────────────────────────────────────────────────────────────────┘

    ┌─────────────────┐           ┌─────────────────────────────────────┐
    │   <<enum>>      │           │              Task                   │
    │   TaskStatus    │           │─────────────────────────────────────│
    │─────────────────│           │ + task_id: str                      │
    │ PENDING         │◄──────────│ + payload: Any                      │
    │ IN_PROGRESS     │           │ + status: TaskStatus                │
    │ COMPLETED       │           │ + result: Optional[Any]             │
    │ FAILED          │           │ + error: Optional[str]              │
    └─────────────────┘           │ + metadata: Dict                    │
                                  │─────────────────────────────────────│
                                  │ + mark_in_progress(worker_id)       │
                                  │ + mark_completed(result)            │
                                  │ + mark_failed(error)                │
                                  └───────────────┬─────────────────────┘
                                                  │
                                                  │ contains
                                                  ▼
┌────────────────────────────────────────────────────────────────────────────┐
│                               TaskQueue                                     │
│────────────────────────────────────────────────────────────────────────────│
│ - _queue: Queue                                                            │
│ - _lock: Lock                                                              │
│ - _tasks: Dict[str, Task]                                                  │
│ - _results: List[Dict]                                                     │
│────────────────────────────────────────────────────────────────────────────│
│ + push(task: Task): None              # Producer method                    │
│ + pop(): Optional[Task]               # Consumer method                    │
│ + report_result(task_id, worker_id, result): None                          │
│ + report_failure(task_id, worker_id, error): None                          │
│ + is_empty(): bool                                                         │
│ + size(): int                                                              │
│ + get_all_results(): List[Dict]                                            │
│ + get_stats(): Dict[str, int]                                              │
└─────────────────────────────────────────────────────────────────────────────┘
         ▲                           ▲                           ▲
         │                           │                           │
         │ uses                      │ uses                      │ uses
         │                           │                           │
┌────────┴────────┐        ┌────────┴────────┐        ┌────────┴────────┐
│   <<abstract>>  │        │   <<abstract>>  │        │   <<abstract>>  │
│    Producer     │        │    Consumer     │        │   Aggregator    │
│─────────────────│        │─────────────────│        │─────────────────│
│ + queue         │        │ + worker_id     │        │ + queue         │
│─────────────────│        │ + queue         │        │─────────────────│
│ + decompose()*  │        │─────────────────│        │ + synthesize()* │
│ + produce()     │        │ + execute()*    │        │ + aggregate()   │
└────────┬────────┘        │ + consume_one() │        └────────┬────────┘
         │                 │ + consume_all() │                 │
         │                 └────────┬────────┘                 │
         │                          │                          │
         │ extends                  │ extends                  │ extends
         │                          │                          │
         ▼                          ▼                          ▼
┌─────────────────┐        ┌─────────────────┐        ┌─────────────────┐
│ QueryDecomposer │        │  SearchWorker   │        │ResultSynthesizer│
│─────────────────│        │─────────────────│        │─────────────────│
│ + decompose()   │        │ + execute()     │        │ + synthesize()  │
└─────────────────┘        └─────────────────┘        └─────────────────┘

                           ┌─────────────────────────────────────────────┐
                           │          WorkflowCoordinator                │
                           │─────────────────────────────────────────────│
                           │ + producer: Producer                        │
                           │ + consumer_factory: Callable                │
                           │ + aggregator: Aggregator                    │
                           │ + num_workers: int                          │
                           │ + queue: TaskQueue                          │
                           │─────────────────────────────────────────────│
                           │ + run(input_data): Any                      │
                           └─────────────────────────────────────────────┘
```

### Sequence Diagram

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                            SEQUENCE DIAGRAM                                 │
└─────────────────────────────────────────────────────────────────────────────┘

  Coordinator     Producer        Queue         Worker1       Worker2      Aggregator
       │              │             │              │             │              │
       │  produce()   │             │              │             │              │
       │─────────────►│             │              │             │              │
       │              │   push(T1)  │              │             │              │
       │              │────────────►│              │             │              │
       │              │   push(T2)  │              │             │              │
       │              │────────────►│              │             │              │
       │              │   push(T3)  │              │             │              │
       │              │────────────►│              │             │              │
       │              │             │              │             │              │
       │──────────────┼─────────────┼──────────────┼─────────────┼──────────────│
       │              │             │  ROUND 1     │             │              │
       │──────────────┼─────────────┼──────────────┼─────────────┼──────────────│
       │              │             │              │             │              │
       │              │             │    pop()     │             │              │
       │              │             │◄─────────────│             │              │
       │              │             │     T1       │             │              │
       │              │             │─────────────►│ execute()   │              │
       │              │             │              │────────┐    │              │
       │              │             │              │        │    │              │
       │              │             │    pop()     │        │    │              │
       │              │             │◄─────────────┼────────┼────│              │
       │              │             │     T2       │        │    │              │
       │              │             │─────────────►│        │    │              │
       │              │             │              │        │    │ execute()    │
       │              │             │              │        │    │─────────┐    │
       │              │             │              │        │    │         │    │
       │              │             │report_result │◄───────┘    │         │    │
       │              │             │◄─────────────│             │         │    │
       │              │             │              │             │         │    │
       │              │             │report_result │             │◄────────┘    │
       │              │             │◄─────────────┼─────────────│              │
       │              │             │              │             │              │
       │──────────────┼─────────────┼──────────────┼─────────────┼──────────────│
       │              │             │  ROUND 2     │             │              │
       │──────────────┼─────────────┼──────────────┼─────────────┼──────────────│
       │              │             │    pop()     │             │              │
       │              │             │◄─────────────│             │              │
       │              │             │     T3       │             │              │
       │              │             │─────────────►│ execute()   │              │
       │              │             │              │────────┐    │              │
       │              │             │report_result │◄───────┘    │              │
       │              │             │◄─────────────│             │              │
       │              │             │              │             │              │
       │  aggregate() │             │              │             │              │
       │─────────────────────────────────────────────────────────────────────►│
       │              │             │ get_results  │             │              │
       │              │             │◄─────────────┼─────────────┼──────────────│
       │              │             │─────────────────────────────────────────►│
       │              │             │              │             │   synthesize │
       │              │             │              │             │              │
       │◄─────────────────────────────────────────────────────────────────────│
       │  final_result│             │              │             │              │
       │              │             │              │             │              │
```

### Key Takeaways

| Concept | Description |
|---------|-------------|
| **Decoupling** | Producer and Consumer are independent; communicate only through queue |
| **Parallelism** | Multiple workers process tasks concurrently |
| **Load Balancing** | Workers self-select tasks; faster workers get more work |
| **Scalability** | Easy to add more workers without changing other components |
| **Thread Safety** | Queue handles synchronization; workers don't need to coordinate |
| **Extensibility** | Abstract base classes allow domain-specific implementations |

### Implementation Checklist

To implement this pattern for your use case:

1. ☐ Define your `Task` structure (what payload represents your work unit?)
2. ☐ Implement `Producer.decompose()` (how to break input into tasks?)
3. ☐ Implement `Consumer.execute()` (how to process one task?)
4. ☐ Implement `Aggregator.synthesize()` (how to combine results?)
5. ☐ Configure `WorkflowCoordinator` with appropriate worker count
6. ☐ Run and iterate!

### Related Patterns

- **Worker Pool + Semaphore**: Pre-allocate workers, submit all tasks at once
- **Fan-Out/Fan-In**: Similar structure, often used in distributed systems
- **Pipeline**: Tasks flow through multiple processing stages
- **Reactor**: Event-driven variation of producer-consumer

---

## Part 9: Extension Points

This section shows how to extend the pattern for common use cases.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXTENSION 1: Priority Queue
# ═══════════════════════════════════════════════════════════════════════════════

from queue import PriorityQueue as PQ
from dataclasses import dataclass, field


@dataclass(order=True)
class PrioritizedTask:
    """
    Task wrapper that enables priority queue ordering.
    Lower priority values are processed first.
    """
    priority: int
    task: Task = field(compare=False)


class PriorityTaskQueue(TaskQueue):
    """
    Extension: Task queue with priority support.
    
    Tasks with lower priority values are processed first.
    Useful when some tasks are more urgent than others.
    """
    
    def __init__(self):
        super().__init__()
        self._queue = PQ()  # Replace FIFO with priority queue
    
    def push(self, task: Task, priority: int = 5) -> None:
        """
        Push a task with priority (lower = more urgent).
        
        Args:
            task: The task to add
            priority: Priority level (1=highest, 10=lowest)
        """
        with self._lock:
            self._tasks[task.task_id] = task
        self._queue.put(PrioritizedTask(priority=priority, task=task))
    
    def pop(self) -> Optional[Task]:
        """Pop highest priority task."""
        try:
            prioritized = self._queue.get_nowait()
            return prioritized.task
        except Exception:
            return None


# ═══════════════════════════════════════════════════════════════════════════════
# EXTENSION 2: Retry Support
# ═══════════════════════════════════════════════════════════════════════════════

class RetryableConsumer(Consumer):
    """
    Extension: Consumer with automatic retry on failure.
    
    Retries failed tasks up to max_retries times before
    marking them as permanently failed.
    """
    
    def __init__(self, worker_id: str, queue: TaskQueue, max_retries: int = 3):
        super().__init__(worker_id, queue)
        self.max_retries = max_retries
    
    def consume_one(self) -> Optional[Dict]:
        """Consume with retry logic."""
        task = self.queue.pop()
        if task is None:
            return None
        
        retries = task.metadata.get("retries", 0)
        task.mark_in_progress(self.worker_id)
        
        try:
            result = self.execute(task.payload)
            self.queue.report_result(task.task_id, self.worker_id, result)
            return {"task_id": task.task_id, "status": "completed", "result": result}
            
        except Exception as e:
            if retries < self.max_retries:
                # Requeue for retry
                task.metadata["retries"] = retries + 1
                task.status = TaskStatus.PENDING
                self.queue.push(task)
                return {"task_id": task.task_id, "status": "retrying", "attempt": retries + 1}
            else:
                # Max retries exceeded
                self.queue.report_failure(task.task_id, self.worker_id, str(e))
                return {"task_id": task.task_id, "status": "failed", "error": str(e)}
    
    @abstractmethod
    def execute(self, payload: Any) -> Any:
        """Execute task - must be implemented by subclass."""
        pass


# ═══════════════════════════════════════════════════════════════════════════════
# EXTENSION 3: Callback Support
# ═══════════════════════════════════════════════════════════════════════════════

class ObservableTaskQueue(TaskQueue):
    """
    Extension: Task queue with event callbacks.
    
    Allows monitoring task lifecycle events for logging,
    metrics, or triggering downstream actions.
    """
    
    def __init__(self):
        super().__init__()
        self._callbacks = {
            "on_push": [],
            "on_pop": [],
            "on_complete": [],
            "on_fail": []
        }
    
    def register_callback(self, event: str, callback) -> None:
        """
        Register a callback for a lifecycle event.
        
        Events: on_push, on_pop, on_complete, on_fail
        """
        if event in self._callbacks:
            self._callbacks[event].append(callback)
    
    def _fire(self, event: str, *args) -> None:
        """Fire all callbacks for an event."""
        for callback in self._callbacks.get(event, []):
            try:
                callback(*args)
            except Exception:
                pass  # Don't let callback errors break the queue
    
    def push(self, task: Task) -> None:
        super().push(task)
        self._fire("on_push", task)
    
    def pop(self) -> Optional[Task]:
        task = super().pop()
        if task:
            self._fire("on_pop", task)
        return task
    
    def report_result(self, task_id: str, worker_id: str, result: Any) -> None:
        super().report_result(task_id, worker_id, result)
        self._fire("on_complete", task_id, worker_id, result)
    
    def report_failure(self, task_id: str, worker_id: str, error: str) -> None:
        super().report_failure(task_id, worker_id, error)
        self._fire("on_fail", task_id, worker_id, error)


# ═══════════════════════════════════════════════════════════════════════════════
# EXTENSION DEMO
# ═══════════════════════════════════════════════════════════════════════════════

print("Extensions loaded successfully!")
print("\nAvailable extensions:")
print("  • PriorityTaskQueue  - Process urgent tasks first")
print("  • RetryableConsumer  - Auto-retry failed tasks")  
print("  • ObservableTaskQueue - Event callbacks for monitoring")

## References

- **Producer-Consumer Pattern**: Wikipedia article on the classic concurrency pattern
- **Python Queue**: Standard library documentation for thread-safe queues
- **Thread Safety**: Python threading module for synchronization primitives
- **Full Implementation**: See `web_search_example_via_task_queue.ipynb` for a complete working example with LangGraph integration